# Appliance states from high-frequency voltage and current

Electrical loads can draw similar average power while producing different current-waveform shapes. This notebook studies whether those waveform signatures, together with their recent history, help identify **which appliances are supplied with power** in aggregate measurements.

**TDHA** is a public household-appliance dataset with intensive switching events. The experiments here use synchronized aggregate voltage/current recordings and socket-switching annotations from controlled appliance combinations. Voltage and current are the inputs; the annotations provide labels. An ON label denotes socket supply state and can include a period when an appliance's internal heating element is inactive.

The notebook implements two tasks:

| Task | Inputs and model | Output | Recorded runs |
|---|---|---|---|
| Microwave state (V2) | Phase-aligned waveform and causal power features, GRU/W1 | One microwave ON-state score per 100 ms evaluation endpoint | Two folds × three seeds |
| Multi-label appliance supply states (V3) | N0-W1 waveform/GRU reference and N1-F harmonic/convolution candidate | One supply-state score for each of 11 appliance-class channels | Two methods × two folds × three seeds |

Each recording monitors **five physical sockets**. Across recordings, socket annotations are mapped to **11 shared appliance-class channels**. The primary AP and Recall summaries use **six classes with shared support**.

The raw waveform sample rate is **51.2 kHz**. V2 uses a **51.2-second history**; V3 uses a **25.6-second history** with 100 ms tokens. Sampling rate, history duration and prediction step describe different parts of the pipeline.

The workflow proceeds from **cycle pairing and ADC conversion → electrical representations and label masks → causal windows and fold normalization → model definitions → training and threshold calibration → cross-record comparison**. The notebook includes the model and training implementations, with full raw-data entries linked below.

All `RUN_*` flags initially equal `False`. Default execution runs small analytic checks and displays recorded results. Data access, model loading, conversion, inference, training and bootstrap require their corresponding flags and inputs. NPU training and temporal hold-out evaluation, CUDA supplemental evaluations and CPU-format weight loading are described separately in the results section.

[Data and dataset sources](../docs/en/data.md) · [Selected V2 model](../src/wattsign/_tdha_waveform.py) · [V3 physics](../npu/tdha_multilabel_v3/tdha_v3/physics.py) · [V3 models](../npu/tdha_multilabel_v3/tdha_v3/models.py) · [V3 training](../npu/tdha_multilabel_v3/tdha_v3/training.py) · [Conversion](../npu/tdha_multilabel_v3/tools/convert_weights.py)


In [ ]:
from __future__ import annotations
import csv, functools, hashlib, io, json, math, os, random, re, sys, time
from collections import Counter, defaultdict, OrderedDict
from datetime import datetime
from pathlib import Path
from zipfile import ZipFile
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, roc_auc_score,
    precision_recall_fscore_support, balanced_accuracy_score, confusion_matrix)
from scipy.stats import spearmanr
sys.dont_write_bytecode = True

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Run from the repository root or notebooks directory")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
NPU_ROOT = ROOT/"npu/tdha_multilabel_v3"
if str(NPU_ROOT) not in sys.path:
    sys.path.insert(0,str(NPU_ROOT))

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

RUN_DATA_READ = False
RUN_MODEL_LOAD = False
RUN_CONVERSION = False
RUN_INFERENCE = False
RUN_TRAINING = False
RUN_BOOTSTRAP = False
STEP_FLAGS = {"raw_sample": "RUN_DATA_READ", "weights": "RUN_MODEL_LOAD",
              "conversion": "RUN_CONVERSION", "inference": "RUN_INFERENCE",
              "training": "RUN_TRAINING", "bootstrap": "RUN_BOOTSTRAP"}
TDHA_ARCHIVE = None      
TDHA_MAPPING = None       # Explicit SocketRecord.xlsx input
TRAIN_OUTPUT = None       # Fresh output directory outside all source checkouts
TRAIN_DEVICE = "cuda"     # Training device: "cpu" or "cuda"
V2_ADC_CACHE = None      
MODEL_FILE = None
MODEL_SHA256 = None      # Trusted value from results/tdha-waveform/source.json
DEVICE = torch.device("cpu") if HAS_TORCH else None
SEEDS = (11, 42, 73)
PAIR_FOLDS = {"A": {"train": (2,4,6), "test": (3,5,7), "cut": (255,340)},
              "B": {"train": (3,5,7), "test": (2,4,6), "cut": (236,315)}}
MICROWAVE_SOCKET = {1:None, 2:4, 3:None, 4:3, 5:4, 6:None, 7:4}
EPOCH = datetime(1970,1,1)
M, CHANNELS = 242, 520
LABELS, TIMELINES, FOLD_FEATURES, HF_RAW, FOLD_NORMS = {}, {}, {}, {}, {}
SAMPLE_MANIFEST, RAW_MMAP, SOURCE_AUDIT = {}, {}, []
POWER5, HIGH_POWER = {}, {}

def manual_step(step):
    def decorate(function):
        @functools.wraps(function)
        def checked(*args, **kwargs):
            if not globals()[STEP_FLAGS[step]]:
                raise RuntimeError(f"{step} is disabled; set {STEP_FLAGS[step]} explicitly")
            if step in {"weights", "conversion", "inference", "training"} and not HAS_TORCH:
                raise ImportError("Install the documented PyTorch environment for this step")
            return function(*args, **kwargs)
        return checked
    return decorate

print("Execution flags: all RUN_* operations disabled.")


In [ ]:
def us_of(s):
    d=datetime.fromisoformat(s.replace("/","-"))-EPOCH
    return (d.days*86400+d.seconds)*1_000_000+d.microseconds


## 1. Pair synchronized cycles

Voltage and current must describe the same instant for power and waveform features to be meaningful. Each paired cycle therefore forms the basic measurement unit before temporal features are constructed.

Channel 1 (voltage) and channel 2 (current) are paired by HomeID, RecvTime and InternalIndex. Duplicate pairs are rejected; each cycle must contain 1024 samples. The bounded iterator reads a specified number of cycles without creating a cache. A cycle spans 20 ms at 51.2 kHz, while predictions use a 100 ms step.


In [ ]:
@manual_step("raw_sample")
def read_paired_cycles(archive, record, limit=1):
    archive = Path(archive)
    if not archive.is_file():
        raise FileNotFoundError("Provide the local TDHA Aggregate.zip; raw data is not bundled")
    if record not in range(1,8) or not 1 <= limit <= 32:
        raise ValueError('Choose one used record (1-7) and at most 32 cycles')
    pending, seen, emitted = {}, set(), 0
    with ZipFile(archive) as z, z.open(f"Aggregate/Aggregation_{record}.csv") as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8-sig", newline="", errors="strict"))
        assert reader.fieldnames == ["id","DeviceTime","HomeID","RecvTime","InternalIndex","Channel","Rate","Value"]
        for row in reader:
            key = (row["HomeID"], row["RecvTime"], row["InternalIndex"])
            ch = int(row["Channel"])
            if ch not in (1,2):
                continue
            pair = pending.setdefault(key,{})
            pair[ch] = row
            if 1 not in pair or 2 not in pair:
                continue
            del pending[key]
            if key in seen:
                continue
            seen.add(key)
            if pair[1]["Rate"] != "1024" or pair[2]["Rate"] != "1024":
                continue
            uv = np.fromstring(pair[1]["Value"], sep="|", dtype=np.int16)
            iv = np.fromstring(pair[2]["Value"], sep="|", dtype=np.int16)
            if uv.size != 1024 or iv.size != 1024:
                continue
            yield {"raw": np.stack((uv,iv)), "time_us": us_of(row["RecvTime"]),
                   "internal_index": int(row["InternalIndex"])}
            emitted += 1
            if emitted >= limit:
                return

if RUN_DATA_READ:
    if TDHA_ARCHIVE is None:
        raise FileNotFoundError("Set TDHA_ARCHIVE explicitly before enabling raw_sample")
    first_cycle = next(read_paired_cycles(TDHA_ARCHIVE, record=2, limit=1))


## 2. ADC conversion and W1 representation

The raw files contain analog-to-digital converter (**ADC**) values. Conversion places voltage and current on their physical scales. W1 then separates the part of current aligned with voltage from the remaining waveform and aligns cycles to a common voltage phase. These transforms give the model electrical structure to learn from while reducing differences caused solely by cycle phase.

ADC scales are `V = ADC/12.90` and `I = ADC×28.47/32768`. W1 adds non-active current `i − <vi>/<v²> v`, aligns each cycle to the voltage fundamental phase and learns a three-channel waveform embedding. Neither labels nor record identity enter these transforms.


In [ ]:
def cycle_quantities(raw):
    assert raw.ndim==3 and raw.shape[1:]==(2,1024)
    u=raw[:,0].astype(np.float32)/12.90
    i=raw[:,1].astype(np.float32)*(28.47/32768.)
    ur=np.sqrt(np.mean(u*u,axis=1,dtype=np.float64))
    ir=np.sqrt(np.mean(i*i,axis=1,dtype=np.float64))
    p=np.mean(u*i,axis=1,dtype=np.float64)
    s=ur*ir
    rad=s*s-p*p
    n_valid=rad>=-1e-5*np.maximum(s*s,1.)
    nonactive=np.sqrt(np.maximum(rad,0.))
    pf_valid=s>1e-5
    pf=np.divide(p,s,out=np.zeros_like(p),where=pf_valid)
    basic=np.stack((p,s,nonactive,ur,ir,pf),axis=1).astype(np.float32)
    bmask=np.stack((np.isfinite(p),np.isfinite(s),n_valid,np.isfinite(ur),np.isfinite(ir),pf_valid),axis=1)
    bmask &= np.isfinite(basic)
    basic[~bmask]=0.
    fu=np.fft.rfft(u,axis=1); fi=np.fft.rfft(i,axis=1)
    phase=np.angle(fu[:,1])
    im=np.maximum(ir,1e-12)
    harmonics=[]
    for h in (3,5,7,9,11):
        z=(np.sqrt(2.)/1024.)*fi[:,h]*np.exp(-1j*h*phase)/im
        harmonics.extend((z.real,z.imag))
    def thd(spec):
        return np.sqrt(np.sum(np.abs(spec[:,2:26])**2,axis=1))/np.maximum(np.abs(spec[:,1]),1e-12)
    central=i-i.mean(axis=1,keepdims=True)
    std=np.sqrt(np.mean(central*central,axis=1))
    kurt=np.mean((central/np.maximum(std[:,None],1e-6))**4,axis=1)
    other=(thd(fi),thd(fu),np.max(np.abs(i),axis=1)/im,kurt,i.mean(axis=1)/im,np.cos(np.angle(fi[:,1])-phase))
    hf=np.stack((*harmonics,*other),axis=1).astype(np.float32)
    assert hf.shape[1]==16
    hmask=np.isfinite(hf)&(ir[:,None]>1e-4)&(ur[:,None]>1.)
    hmask[:,11] &= np.abs(fi[:,1])>1e-3
    hmask[:,10] &= np.abs(fi[:,1])>1e-3
    hmask[:,11] &= np.abs(fu[:,1])>1e-3
    hmask[:,15] &= (np.abs(fi[:,1])>1e-3)&(np.abs(fu[:,1])>1e-3)
    hmask[:,13] &= std>1e-4
    hf[~hmask]=0.
    return basic,bmask.astype(np.uint8),hf,hmask.astype(np.uint8)

def physical_wave(raw):
    u=raw[:,0].astype(np.float32)/12.90
    i=raw[:,1].astype(np.float32)*(28.47/32768.)
    p=np.mean(u*i,axis=1,dtype=np.float64)
    u2=np.mean(u*u,axis=1,dtype=np.float64)
    good=np.isfinite(u2)&(u2>1e-12)
    f=i-(np.divide(p,u2,out=np.zeros_like(p),where=good)[:,None]*u).astype(np.float32)
    return u,i,f,good

def align_one_cycle(u):
    ph=np.angle(np.fft.rfft(u)[1])
    shift=int(np.rint(ph*1024/(2*np.pi)))
    return np.roll(u,shift)

def f_current(raw):
    u,i,_,good=physical_wave(raw)
    ur=np.sqrt(np.mean(u*u,axis=1,dtype=np.float64))
    ir=np.sqrt(np.mean(i*i,axis=1,dtype=np.float64))
    uf=np.fft.rfft(u,axis=1)[:,1]
    ff=np.fft.rfft(i,axis=1)[:,1:26]
    h=np.arange(1,26)
    rot=np.exp(-1j*np.angle(uf)[:,None]*h[None,:])
    amp=np.sqrt(2.)/1024*ff*rot
    ab=np.stack((amp.real,amp.imag),axis=-1).reshape(-1,50)
    rel=ab/np.maximum(ir[:,None],1e-12)
    valid=np.isfinite(ab).all(1)&np.isfinite(uf)&(ur>1.)&(np.abs(uf)>1e-3)&good
    relvalid=valid&(ir>1e-6)
    vals=np.concatenate((ab,rel),axis=1).astype(np.float32)
    vals[~valid,:50]=0;vals[~relvalid,50:]=0
    mask=np.stack((valid,relvalid),axis=1).astype(np.uint8)
    return vals,mask


In [ ]:
# Analytic illustration only: four generated cycles, no dataset access.
theta = 2*np.pi*np.arange(1024)/1024
u_demo = 325*np.sin(theta)
i_demo = np.sin(theta) + 0.2*np.sin(3*theta)
adc_demo = np.rint(np.stack((u_demo*12.90, i_demo*32768/28.47))).astype(np.int16)[None]
basic_demo, basic_mask_demo, hf_demo, hf_mask_demo = cycle_quantities(adc_demo)
u_demo_si, i_demo_si, nonactive_demo, valid_demo = physical_wave(adc_demo)
assert basic_mask_demo.all() and np.isclose(basic_demo[0,0],162.5,rtol=.03)
assert np.isclose(hf_demo[0,10],.2,atol=.02)
assert np.allclose(align_one_cycle(np.roll(u_demo,137)), align_one_cycle(u_demo), atol=1e-10)
assert np.max(np.abs((u_demo/325) - (np.mean(u_demo*(u_demo/325))/np.mean(u_demo*u_demo))*u_demo)) < 1e-6
pd.DataFrame(basic_demo, columns=["P","S","nonactive","Vrms","Irms","power_factor"])


## 3. Labels, continuity and causal windows

V2 projects the latest known label, excludes ±150 ms around target transitions and splits history at continuity or partition boundaries. Eligible windows include the full history and baseline warm-up. Baselines use `shift(1).rolling(...)` so only past samples contribute. Training endpoints occur every second and evaluation endpoints every 100 ms; the selected history is 2560 cycles (51.2 s).

The functions operate on `LABELS` and `TIMELINES` supplied by data preparation.


In [ ]:
def socket_mapping(path):
    ns={"m":"http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with ZipFile(path) as z:
        root=ET.fromstring(z.read("xl/sharedStrings.xml"))
        shared=["".join(t.text or "" for t in e.findall(".//m:t",ns)) for e in root.findall("m:si",ns)]
        result={}
        for rec in range(1,8):
            sheet=ET.fromstring(z.read(f"xl/worksheets/sheet{rec}.xml"))
            rows=[]
            for row in sheet.findall(".//m:sheetData/m:row",ns):
                dd={}
                for cell in row.findall("m:c",ns):
                    v=cell.find("m:v",ns)
                    if v is None: continue
                    col=re.match("[A-Z]+",cell.attrib["r"]).group()
                    dd[col]=shared[int(v.text)] if cell.attrib.get("t")=="s" else v.text
                rows.append(dd)
            microw=[r for r in rows if r.get("A","").strip().lower()=="microwave"]
            if not microw: result[rec]=None
            else:
                assert len(microw)==1
                marks=[i for i,c in enumerate("BCDEF") if c in microw[0] and microw[0][c] not in ("0","")]
                assert len(marks)<=1,(rec,microw[0])
                result[rec]=marks[0] if marks else None
    return result

def parse_state(s):
    pairs=re.findall(r"Socket([0-4]):\s*([01])",s)
    assert len(pairs)==5 and set(int(k) for k,_ in pairs)==set(range(5)),s
    d={int(k):int(v) for k,v in pairs}
    return tuple(d[i] for i in range(5))

def load_labels(z,rec):
    with z.open(f"Aggregate/Event/Aggregation_sign_{rec}.csv") as f:
        rows=list(csv.DictReader(io.TextIOWrapper(f,encoding="utf-8-sig",newline="")))
    times=np.asarray([us_of(r["Time"]) for r in rows],dtype=np.int64)
    states=np.asarray([parse_state(r["event"]) for r in rows],dtype=np.uint8)
    assert np.all(np.diff(times)>0) and len(times)>300
    target=np.zeros(len(times),np.uint8) if MICROWAVE_SOCKET[rec] is None else states[:,MICROWAVE_SOCKET[rec]]
    if MICROWAVE_SOCKET[rec] is None: assert not target.any()
    changes=np.r_[False,target[1:]!=target[:-1]]
    any_changes=np.r_[False,np.any(states[1:]!=states[:-1],axis=1)]
    return {"time":times,"states":states,"target":target,"change_time":times[changes],"all_change_time":times[any_changes],"n_labels":len(times),"on_switches":int(np.sum((target[1:]==1)&(target[:-1]==0))),"off_switches":int(np.sum((target[1:]==0)&(target[:-1]==1)))}

def make_labels(rec,times):
    lab=LABELS[rec]; ix=np.searchsorted(lab["time"],times,side="right")-1
    known=(ix>=0)&(times<=lab["time"][-1])
    y=np.zeros(len(times),np.uint8)
    y[known]=lab["target"][ix[known]]
    changes=lab["change_time"]
    if len(changes):
        pos=np.searchsorted(changes,times)
        left=changes[np.maximum(pos-1,0)]
        right=changes[np.minimum(pos,len(changes)-1)]
        dist=np.minimum(np.abs(times-left),np.abs(times-right))
        known &= dist>150_000
    else: dist=np.full(len(times),np.iinfo(np.int64).max,dtype=np.int64)
    event=np.full(len(times),-1,np.int32)
    starts=np.flatnonzero((lab["target"]==1)&np.r_[True,lab["target"][:-1]==0])
    if len(starts):
        ords=np.searchsorted(lab["time"][starts],times,side="right")-1
        active=(y==1)&(ords>=0)
        event[active]=rec*1000+ords[active].astype(np.int32)
    any_ch=lab["all_change_time"]
    pos=np.searchsorted(any_ch,times)
    near=np.minimum(np.abs(times-any_ch[np.maximum(pos-1,0)]),np.abs(times-any_ch[np.minimum(pos,len(any_ch)-1)])) if len(any_ch) else np.full(len(times),10**15,dtype=np.int64)
    return y,known.astype(np.uint8),event,dist,near

def phase_of(rec,fold):
    lab=LABELS[rec]; tt=TIMELINES[rec]["time"]
    in_span=(tt>=lab["time"][0])&(tt<=lab["time"][-1])
    phase=np.full(len(tt),-1,np.int8)
    if rec in PAIR_FOLDS[fold]["train"]:
        ca,cb=PAIR_FOLDS[fold]["cut"]
        cuts=lab["time"][[ca,cb]]
        phase[in_span]=np.searchsorted(cuts,tt[in_span],side="right").astype(np.int8)
    else:phase[in_span]=3
    return phase

def segment_of(rec,fold):
    arr=TIMELINES[rec];phase=phase_of(rec,fold)
    n=len(phase); change=np.r_[True,(phase[1:]!=phase[:-1])|(arr["continuity"][1:]!=arr["continuity"][:-1])]
    seg=np.cumsum(change,dtype=np.int32)
    start=np.maximum.accumulate(np.where(change,np.arange(n,dtype=np.int64),0))
    return phase,seg,start

def causal_feature_block(vals,masks,seg):
    n,k=vals.shape; assert masks.shape==(n,k)
    out=np.zeros((n,k*6),np.float32)
    for j in range(k):
        base=vals[:,j].astype(np.float64); valid=masks[:,j].astype(bool)
        left=np.r_[False,(seg[1:]==seg[:-1])&(valid[1:])&(valid[:-1])]
        delta=np.zeros(n,np.float64);delta[1:]=np.where(left[1:],base[1:]-base[:-1],0.)
        rolling=pd.Series(np.where(valid,base,np.nan)).groupby(seg,sort=False).transform(
            lambda x:x.shift(1).rolling(50,min_periods=42).median()).to_numpy()
        bvalid=valid&np.isfinite(rolling)
        out[:,6*j]=np.where(valid,base,0.)
        out[:,6*j+1]=delta
        out[:,6*j+2]=np.where(bvalid,base-rolling,0.)
        out[:,6*j+3]=valid
        out[:,6*j+4]=left
        out[:,6*j+5]=bvalid
    return out

def endpoint_index(rec,fold,part,train=False):
    arr=TIMELINES[rec];phase,seg,start=segment_of(rec,fold)
    idx=np.arange(len(phase),dtype=np.int64)
    cadence=(idx%50==49) if train else (idx%5==4)
    ok=(phase==part)&(arr["known"]==1)&cadence&(idx-start>=2609)&(arr["bmask"][:,0]==1)
    ep=idx[ok]
    assert not len(ep) or np.all(seg[ep]==seg[ep-2559])
    assert not len(ep) or np.all(seg[ep]==seg[ep-2609])
    return ep


In [ ]:
# Future perturbation and segment boundary check on generated values.
values_demo = np.arange(120, dtype=np.float32)[:,None]
mask_demo = np.ones_like(values_demo, dtype=bool)
segment_demo = np.r_[np.zeros(60,int),np.ones(60,int)]
causal_demo = causal_feature_block(values_demo, mask_demo, segment_demo)
changed_demo = values_demo.copy(); changed_demo[90:] += 999
assert np.array_equal(causal_demo[:90], causal_feature_block(changed_demo,mask_demo,segment_demo)[:90])
assert causal_demo[60,4] == 0  # Delta mask is reset at a new segment.
assert causal_demo[60,5] == 0  # Rolling baseline does not cross the boundary.
print("ADC, alignment, causal-prefix and segment-boundary checks passed.")


## 4. Fold normalization and endpoint tensors

Fold-specific `normalizer.npz` files store training means/stds and voltage/current scales. Tensor construction uses 36 causal power values/masks plus reserved representation slots, with W1 channels and masks in the checkpoint's expected order. New training fits these parameters within each fold's training partition.


In [ ]:
def apply_saved_power_normalizer(values, masks, segments, normalizer):
    base = causal_feature_block(values, masks, segments)
    for j in range(6):
        for k in range(3):
            v = base[:,6*j+k]; ok = base[:,6*j+3+k].astype(bool); q = j*3+k
            v[ok] = (v[ok]-normalizer["base_mean"][q])/normalizer["base_std"][q]
            v[~ok] = 0.
    return base

@manual_step("raw_sample")
def raw_memmap(rec):
    if V2_ADC_CACHE is None:
        raise FileNotFoundError("Set V2_ADC_CACHE to an existing read-only cycle cache")
    if rec not in RAW_MMAP:
        n = SOURCE_AUDIT[rec-1]["cycles"]
        path = Path(V2_ADC_CACHE)/f"agg{rec}_adc_int16.bin"
        if not path.is_file() or path.stat().st_size != n*2*1024*2:
            raise FileNotFoundError("Missing or inconsistent selected ADC cache")
        RAW_MMAP[rec] = np.memmap(path,dtype=np.int16,mode="r",shape=(n,2,1024))
    return RAW_MMAP[rec]


In [ ]:
def window_arrays(fold,rec,eps,rep,length=640):
    if rep != "W1": raise ValueError("The V2 model adapter supports W1.")
    eps=np.asarray(eps,dtype=np.int64);x=np.zeros((len(eps),CHANNELS,length),np.float32)
    feat=FOLD_FEATURES[fold][rec];source=HF_RAW[rec]
    for k,ep in enumerate(eps):
        ix=np.arange(ep-length+1,ep+1,dtype=np.int64)
        assert ix[0]>=feat["start"][ep] and feat["segment"][ix[0]]==feat["segment"][ep]
        x[k,:36]=feat["base"][ix].T
        x[k,36+M:36+M+16]=source["fm"][ix,0].astype(np.float32)[None]
    return x

def pairs_y(pairs):return np.asarray([TIMELINES[int(r)]["y"][int(e)] for r,e in pairs],np.uint8)

def chunk_features(fold,pairs,rep,length):
    x=np.empty((len(pairs),CHANNELS,length),np.float32)
    for rec in np.unique(pairs[:,0]):
        sel=np.flatnonzero(pairs[:,0]==rec)
        x[sel]=window_arrays(fold,int(rec),pairs[sel,1],rep,length)
    return x

def chunk_raw(pairs,length):
    return np.stack([raw_memmap(int(r))[int(e)-length+1:int(e)+1] for r,e in pairs])


## 5. GRU/W1 architecture

A gated recurrent unit (**GRU**) combines information across a sequence. Here, the waveform encoder first describes each short electrical cycle; pooling and the causal stem summarize cycles before the GRU uses their history to predict the microwave state at the endpoint.

The GRU/W1/L2560 model uses a three-channel waveform encoder, mean/max cycle pooling, a causal segment stem, a two-layer 48-unit GRU and an endpoint classifier. Attribute names and forward computations match the saved checkpoints. The model adapter accepts this configuration.


In [ ]:
if HAS_TORCH:
    class CausalConv(nn.Module):

        def __init__(self, a, b, k=3, d=1):
            super().__init__()
            self.pad = (k - 1) * d
            self.conv = nn.Conv1d(a, b, k, dilation=d)

        def forward(self, x):
            return self.conv(F.pad(x, (self.pad, 0)))

    class SegmentStem(nn.Module):

        def __init__(self):
            super().__init__()
            self.conv = CausalConv(CHANNELS, 32, 5)
            self.proj = nn.Linear(64, 32)
            self.norm = nn.LayerNorm(32)

        def forward(self, x):
            z = F.relu(self.conv(x))
            n = z.shape[-1]
            assert n % 5 == 0
            z = z.reshape(len(z), 32, n // 5, 5)
            return self.norm(self.proj(torch.cat((z.mean(-1), z.amax(-1)), dim=1).transpose(1, 2)))

    class RecurrentTime(nn.Module):

        def __init__(self, family, length):
            super().__init__()
            self.stem = SegmentStem()
            layer = nn.GRU if family == 'GRU' else nn.LSTM
            self.memory = layer(32, 48, 2, batch_first=True, dropout=0.1)
            self.head = nn.Sequential(nn.Linear(48, 16), nn.ReLU(), nn.Dropout(0.1), nn.Linear(16, 1))

        def forward(self, x):
            out, _ = self.memory(self.stem(x))
            return self.head(out[:, -1]).squeeze(-1)

    class W1Encoder(nn.Module):

        def __init__(self):
            super().__init__()
            self.conv = nn.Sequential(nn.Conv1d(3, 8, 15, stride=4, padding=7), nn.ReLU(), nn.Conv1d(8, 16, 9, stride=4, padding=4), nn.ReLU(), nn.Conv1d(16, 16, 5, stride=2, padding=2), nn.ReLU())
            self.out = nn.Linear(256, 16)

        def forward(self, w):
            z = self.conv(w)
            assert z.shape[-1] == 32
            z = z.reshape(len(z), 16, 8, 4)
            pooled = torch.cat((z.mean(-1), z.amax(-1)), dim=1).flatten(1)
            return self.out(pooled)

    class AggregateModel(nn.Module):

        def __init__(self, family, rep, length, vscale, iscale):
            super().__init__()
            self.family = family
            self.rep = rep
            self.length = length
            if (family, rep, length) != ('GRU', 'W1', 2560):
                raise ValueError('The public adapter supports the selected GRU/W1/2560 model only')
            self.temporal = RecurrentTime('GRU', length)
            self.encoder = W1Encoder()
            self.register_buffer('vscale', torch.tensor(float(vscale)))
            self.register_buffer('iscale', torch.tensor(float(iscale)))

        def encode_cycles(self, raw):
            n = raw.shape[0]
            a = raw.reshape(-1, 2, 1024)
            u = a[:, 0].float() / 12.9
            i = a[:, 1].float() * (28.47 / 32768.0)
            p = (u * i).mean(-1)
            u2 = (u * u).mean(-1).clamp_min(1e-12)
            f = i - p[:, None] / u2[:, None] * u
            phase = torch.angle(torch.fft.rfft(u, dim=-1)[:, 1])
            shifts = torch.round(phase * 1024 / (2 * math.pi)).long()
            positions = (torch.arange(1024, device=u.device)[None] - shifts[:, None]) % 1024
            inp = torch.stack((u / self.vscale, i / self.iscale, f / self.iscale), dim=1)
            inp = torch.gather(inp, 2, positions[:, None].expand(-1, 3, -1))
            return self.encoder(inp).reshape(n, -1, 16)

        def forward(self, x, raw=None, embedding=None):
            if self.encoder is not None:
                if embedding is None:
                    assert raw is not None
                    embedding = self.encode_cycles(raw)
                x = x.clone()
                mask = x[:, 36 + M:36 + M + 16].clone()
                x[:, 36:52] = embedding.transpose(1, 2) * mask
            return self.temporal(x)
else:
    print("PyTorch is required for model construction.")


## 6. Optimization and readout

V2 uses AdamW (`lr=1e-3`, `weight_decay=1e-4`), effective batch size 64 accumulated in microbatches of 8, at most 40 epochs and patience 5. Early stopping minimizes balanced calibration BCE. Sampling and batch orders are fixed by fold and seed. The scorer restores model mode and checks that evaluation leaves the training RNG unchanged.

The training entry below prepares causal features, labels, fold-specific normalizers and sample orders from local raw files, then saves a separate run.


In [ ]:
def fix_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def rng_digest():
    h=hashlib.sha256()
    h.update(torch.get_rng_state().numpy().tobytes())
    for q in torch.cuda.get_rng_state_all(): h.update(q.cpu().numpy().tobytes())
    return h.hexdigest()

def model_state_hash(model):
    h=hashlib.sha256()
    for k,v in sorted(model.state_dict().items()):
        h.update(k.encode());h.update(np.ascontiguousarray(v.detach().cpu().numpy()).tobytes())
    return h.hexdigest()

def binary_bce(y,logits):
    y=np.asarray(y);z=np.asarray(logits,dtype=np.float64)
    loss=np.logaddexp(0.,z)-y*z
    assert (y==0).any() and (y==1).any()
    return float(.5*(loss[y==0].mean()+loss[y==1].mean()))

def feature_timeline(fold,rec,rep):
    if rep != "W1": raise ValueError("The V2 model adapter supports W1.")
    src=FOLD_FEATURES[fold][rec];n=len(src["base"])
    x=np.zeros((CHANNELS,n),np.float32);x[:36]=src["base"].T
    x[36+M:36+M+16]=HF_RAW[rec]["fm"][:,0][None]
    return x

@manual_step('inference')
def encode_full_record(model,rec,batch=512):
    assert model.encoder is not None
    mm=raw_memmap(rec);out=np.empty((len(mm),16),np.float32)
    with torch.inference_mode():
        for lo in range(0,len(mm),batch):
            raw=torch.from_numpy(np.asarray(mm[lo:lo+batch]).copy()).to(DEVICE)
            out[lo:lo+len(raw)]=model.encode_cycles(raw[:,None]).reshape(-1,16).float().cpu().numpy()
    return out

@manual_step('inference')
def score_neural(model, fold, pairs, batch=64):
    if (model.family, model.rep, model.length) != ('GRU', 'W1', 2560):
        raise ValueError('Only selected GRU/W1/L2560 readout is supported')
    prior = rng_digest()
    old = model.training
    model.eval()
    out = np.empty(len(pairs), np.float32)
    try:
        with torch.inference_mode():
            for rec in np.unique(pairs[:, 0]):
                sel = np.flatnonzero(pairs[:, 0] == rec)
                ep = pairs[sel, 1].astype(np.int64)
                xx = feature_timeline(fold, int(rec), model.rep)
                emb = encode_full_record(model, int(rec)) if model.encoder is not None else None
                if emb is not None:
                    xx[36:52] = emb.T * xx[36 + M:36 + M + 16]
                tensor = torch.from_numpy(np.ascontiguousarray(xx[None])).to(DEVICE)
                stem = model.temporal.stem
                z = F.relu(stem.conv(tensor))
                usable = z.shape[-1] // 5 * 5
                z = z[:, :, :usable].reshape(1, 32, usable // 5, 5)
                tokens = stem.norm(stem.proj(torch.cat((z.mean(-1), z.amax(-1)), dim=1).transpose(1, 2)))[0]
                for lo in range(0, len(ep), batch):
                    ids = ep[lo:lo + batch]
                    start = ids - model.length + 1
                    assert np.all(start % 5 == 0)
                    positions = torch.as_tensor(start[:, None] // 5 + np.arange(model.length // 5)[None, :], device=DEVICE)
                    group = tokens[positions].clone()
                    first = np.stack([xx[:, v:v + 5] for v in start])
                    first = torch.from_numpy(first).to(DEVICE)
                    group[:, 0] = stem(first)[:, 0]
                    h = model.temporal.memory(group)[0][:, -1]
                    out[sel[lo:lo + len(ids)]] = model.temporal.head(h).squeeze(-1).float().cpu().numpy()
                del tensor, z, tokens
                del xx, emb
        return out
    finally:
        model.train(old)
        assert rng_digest() == prior, 'Evaluation altered training RNG state'

@manual_step('training')
def fit_neural(fold,seed,family,rep,length,calpairs,cal_y):
    if not RUN_INFERENCE or not RUN_DATA_READ:
        raise RuntimeError("Training also requires explicit inference and raw_sample switches")
    fix_seed(seed)
    norm=FOLD_NORMS[fold]
    model=AggregateModel(family,rep,length,norm["voltage_scale"],norm["current_scale"]).to(DEVICE)
    init_hash=model_state_hash(model.temporal)
    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    sample=SAMPLE_MANIFEST[fold][seed];pairs=sample["pairs"];y=pairs_y(pairs).astype(np.float32)
    best=None;best_loss=np.inf;best_epoch=0;bad=0;history=[];t0=time.time()
    for epoch in range(40):
        model.train();order=sample["order"][epoch];total=0.
        for left in range(0,len(order),64):
            group=order[left:left+64];den=len(group);optimizer.zero_grad(set_to_none=True)
            for j in range(0,den,8):
                ids=group[j:j+8];batchpairs=pairs[ids]
                x=torch.from_numpy(chunk_features(fold,batchpairs,rep,length)).to(DEVICE)
                raw=torch.from_numpy(chunk_raw(batchpairs,length).copy()).to(DEVICE) if rep in ("W0","W1") else None
                logit=model(x,raw=raw)
                target=torch.from_numpy(y[ids]).to(DEVICE)
                loss=F.binary_cross_entropy_with_logits(logit,target,reduction="sum")/den
                loss.backward();total+=float(loss.detach().cpu())*den
                del x,raw,logit,target,loss
            optimizer.step()
        scores=score_neural(model,fold,calpairs)
        cal_loss=binary_bce(cal_y,scores)
        history.append({"epoch":epoch+1,"sample_BCE":total/len(order),"balanced_cal_BCE":cal_loss})
        print(f"{fold} seed{seed} {family}-{rep} L{length} epoch{epoch+1}: {cal_loss:.5f}",flush=True)
        if cal_loss<best_loss-1e-8:
            best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            best_loss=cal_loss;best_epoch=epoch+1;bad=0
        else:bad+=1
        if bad>=5:break
    model.load_state_dict(best);model.eval()
    return model,history,best_epoch,init_hash,round(time.time()-t0,2)


## 7. Metrics and paired bootstrap

Metrics distinguish absent-positive records, ordinary OFF false alarms, high-power OFF false alarms and power-matched evidence. Evaluation uses calibration thresholds and saved high-power cutoffs. Paired bootstrap uses 60 s blocks with shared resample counts. The result section reports the recorded confidence intervals; bootstrap execution requires `RUN_BOOTSTRAP`.


In [ ]:
def power5(rec):
    p=TIMELINES[rec]["basic"][:,0].astype(np.float64)
    cs=np.r_[0.,np.cumsum(p)]
    z=np.full(len(p),np.nan,np.float32)
    z[4:]=((cs[5:]-cs[:-5])/5.).astype(np.float32)
    return z

def sigmoid(x):
    x=np.asarray(x,np.float64)
    return 1/(1+np.exp(-np.clip(x,-100,100)))

def condition_50w(y,p,event,score):
    y=np.asarray(y);p=np.asarray(p);event=np.asarray(event);score=np.asarray(score)
    pos=np.flatnonzero((y==1)&(event>=0));all_events=np.unique(event[pos])
    if not len(pos):return np.nan,0.,{}
    off=np.flatnonzero(y==0);bins=np.floor(p/50).astype(np.int64)
    ev_value=defaultdict(list)
    for bin_ in np.unique(bins[pos]):
        ref=off[bins[off]==bin_]
        if not len(ref):continue
        ordered=np.sort(score[ref])
        ids=pos[bins[pos]==bin_]
        left=np.searchsorted(ordered,score[ids],side="left")
        right=np.searchsorted(ordered,score[ids],side="right")
        for j,val in zip(ids,(left+right)/(2*len(ref))):
            ev_value[int(event[j])].append(float(val))
    event_mean={k:float(np.mean(v)) for k,v in ev_value.items()}
    return (float(np.mean(list(event_mean.values()))) if event_mean else np.nan,
            len(event_mean)/len(all_events),event_mean)

def record_metrics(rec,fold,logit,threshold,endpoint,stage):
    a=TIMELINES[rec];y=a["y"][endpoint];p=POWER5[rec][endpoint]
    prob=sigmoid(logit);pred=np.asarray(logit)>=threshold
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    has_two=len(np.unique(y))==2
    precision,recall,f1,_=precision_recall_fscore_support(y,pred,average="binary",zero_division=0)
    cond,coverage,ev=condition_50w(y,p,a["event"][endpoint],logit)
    off=y==0;high=off&(p>=HIGH_POWER[fold])
    steady=a["distance_any_us"][endpoint]>1_000_000
    safe500=a["distance_switch_us"][endpoint]>500_000
    ys=y[steady];ps=pred[steady]
    return {"record":rec,"fold":fold,"stage":stage,"n":len(endpoint),"positive":int(y.sum()),"prevalence":float(y.mean()),
            "AP":float(average_precision_score(y,logit)) if has_two else np.nan,
            "ROC_AUC":float(roc_auc_score(y,logit)) if has_two else np.nan,
            "precision":float(precision),"recall":float(recall) if y.any() else np.nan,"F1":float(f1) if y.any() else np.nan,
            "balanced_accuracy":float(balanced_accuracy_score(y,pred)) if has_two else np.nan,
            "TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp),
            "off_FPR":float(fp/off.sum()) if off.any() else np.nan,
            "high_off_FPR":float(pred[high].mean()) if high.any() else np.nan,
            "conditional_50w":cond,"condition_coverage":coverage,
            "off_score_P_spearman":float(spearmanr(logit[off],p[off]).statistic) if off.sum()>2 else np.nan,
            "steady_AP":float(average_precision_score(ys,logit[steady])) if len(np.unique(ys))==2 else np.nan,
            "steady_ON_recall":float(ps[ys==1].mean()) if np.any(ys==1) else np.nan,
            "steady_OFF_FPR":float(ps[ys==0].mean()) if np.any(ys==0) else np.nan,
            "mask_500ms_n":int(safe500.sum()),
            "mask_500ms_AP":float(average_precision_score(y[safe500],logit[safe500])) if len(np.unique(y[safe500]))==2 else np.nan,
            "mask_500ms_recall":float(pred[safe500 & (y==1)].mean()) if np.any(safe500 & (y==1)) else np.nan,
            "mask_500ms_off_FPR":float(pred[safe500 & (y==0)].mean()) if np.any(safe500 & (y==0)) else np.nan}

def per_record_metrics(fold,pairs,y,logit,threshold,stage):
    out=[]
    for rec in np.unique(pairs[:,0]):
        select=np.flatnonzero(pairs[:,0]==rec)
        eps=pairs[select,1].astype(np.int64)
        assert np.array_equal(TIMELINES[int(rec)]["y"][eps],y[select])
        out.append(record_metrics(int(rec),fold,logit[select],threshold,eps,stage))
    return out

@manual_step('bootstrap')
def block_plan(fold,stage,rec,endpoint,n=2000):
    times=TIMELINES[rec]["time"][endpoint]
    raw=((times-times[0])//60_000_000).astype(np.int64)
    ids,bcode=np.unique(raw,return_inverse=True)
    y=TIMELINES[rec]["y"][endpoint]
    pos=np.bincount(bcode,weights=y,minlength=len(ids))
    neg=np.bincount(bcode,weights=1-y,minlength=len(ids))
    rng=np.random.default_rng(42+rec+10*(fold=="B")+100*(stage=="time"))
    counts=[];attempts=0
    while len(counts)<n and attempts<n*20:
        attempts+=1
        c=np.bincount(rng.integers(len(ids),size=len(ids)),minlength=len(ids)).astype(np.uint16)
        if ((not np.any(y==1)) or (c@pos)>0) and ((not np.any(y==0)) or (c@neg)>0):counts.append(c)
    assert len(counts)==n,(fold,stage,rec,len(counts))
    return bcode.astype(np.int16),np.stack(counts),attempts-n

@manual_step('bootstrap')
def ap_from_block_counts(y,score,code,counts):
    y=np.asarray(y,np.uint8);score=np.asarray(score,np.float32)
    order=np.argsort(-score,kind="stable")
    yy=y[order];ss=score[order];cc=code[order]
    starts=np.r_[0,np.flatnonzero(ss[:-1]!=ss[1:])+1]
    ends=np.r_[starts[1:]-1,len(ss)-1]
    result=np.empty(len(counts),np.float32)
    for lo in range(0,len(counts),32):
        w=counts[lo:lo+32][:,cc].astype(np.float32)
        posw=w*yy[None,:]
        cum=np.cumsum(w,axis=1,dtype=np.float32)
        poscum=np.cumsum(posw,axis=1,dtype=np.float32)
        group_pos=poscum[:,ends]-np.where(starts[None,:]>0,poscum[:,np.maximum(starts-1,0)],0)
        ratio=np.divide(poscum[:,ends],cum[:,ends],out=np.zeros_like(group_pos),where=cum[:,ends]>0)
        result[lo:lo+len(w)]=(ratio*group_pos).sum(axis=1)/poscum[:,-1]
    return result

@manual_step('bootstrap')
def ratio_from_block_counts(numerator,denominator,code,counts):
    n=np.bincount(code,weights=numerator,minlength=counts.shape[1])
    d=np.bincount(code,weights=denominator,minlength=counts.shape[1])
    nr=counts@n;dr=counts@d
    return np.divide(nr,dr,out=np.full(len(counts),np.nan),where=dr>0).astype(np.float32)


## 8. Multi-label representations and NPU architectures

The multi-label task predicts several socket states at once from a shared aggregate signal. The reference learns directly from phase-aligned waveform samples; the candidate uses harmonic features that describe the amplitudes and phases of frequency components. Comparing them tests a representation-and-architecture choice under a changed recording script.

V3 combines causal power features with a high-frequency branch using the same ADC scales. N0-W1 uses phase-aligned waveforms and an explicit FP32 GRU. N1-F uses phase-referenced harmonics and depthwise causal large/small kernels. Notebook class names have a `Npu` prefix to distinguish them from V2; state-dictionary attribute names match `tdha_v3`.


In [ ]:
def physical(raw):
    assert raw.ndim==3 and raw.shape[1:]==(2,1024)
    u=raw[:,0].astype(np.float32)/12.90
    i=raw[:,1].astype(np.float32)*(28.47/32768.)
    p=np.mean(u*i,axis=1,dtype=np.float64)
    u2=np.mean(u*u,axis=1,dtype=np.float64)
    good=np.isfinite(u2)&(u2>1e-12)
    non=i-(np.divide(p,u2,out=np.zeros_like(p),where=good)[:,None]*u).astype(np.float32)
    return u,i,non,good

def harmonics(raw):
    u,i,_,good=physical(raw)
    ur=np.sqrt(np.mean(u*u,axis=1,dtype=np.float64)); ir=np.sqrt(np.mean(i*i,axis=1,dtype=np.float64))
    uf=np.fft.rfft(u,axis=1)[:,1]; fi=np.fft.rfft(i,axis=1)[:,1:26]
    a=np.sqrt(2.)/1024*fi*np.exp(-1j*np.angle(uf)[:,None]*np.arange(1,26))
    ab=np.stack((a.real,a.imag),-1).reshape(-1,50)
    valid=np.isfinite(ab).all(1)&np.isfinite(uf)&(ur>1.)&(np.abs(uf)>1e-3)&good
    relvalid=valid&(ir>1e-6)
    val=np.concatenate((ab,ab/np.maximum(ir[:,None],1e-12)),1).astype(np.float32)
    mask=np.concatenate((np.repeat(valid[:,None],50,1),np.repeat(relvalid[:,None],50,1)),1)
    return np.where(mask,val,0),mask

def aligned(raw):
    u,i,f,g=physical(raw)
    phase=np.angle(np.fft.rfft(u,axis=1)[:,1]); shifts=np.rint(phase*1024/(2*np.pi)).astype(int)
    ix=(np.arange(1024)[None,:]-shifts[:,None])%1024
    out=np.stack([np.take_along_axis(x,ix,1) for x in (u,i,f)],1)
    out[~g]=0
    return out,g

def changes(values,mask,seg):
    n,k=values.shape; out=np.zeros((n,k*3),np.float32); valid=np.zeros_like(out,bool)
    for j in range(k):
        v=values[:,j].astype(np.float64); m=mask[:,j].astype(bool)
        dm=np.r_[False,(seg[1:]==seg[:-1])&m[1:]&m[:-1]]
        baseline=pd.Series(np.where(m,v,np.nan)).groupby(seg,sort=False).transform(lambda a:a.shift(1).rolling(50,min_periods=42).median()).to_numpy()
        bm=m&np.isfinite(baseline)
        out[:,3*j]=np.where(m,v,0)
        out[1:,3*j+1]=np.where(dm[1:],np.diff(v),0)
        out[:,3*j+2]=np.where(bm,v-baseline,0)
        valid[:,3*j:3*j+3]=np.stack((m,dm,bm),1)
    return out,valid

def frequency_changes(v,m,seg):
    n=len(v); x=np.zeros((n,200),np.float32); mask=np.zeros((n,200),bool)
    x[:,:100]=v; mask[:,:100]=m
    for lag,slot in ((5,100),(50,150)):
        ok=m[lag:,:50]&m[:-lag,:50]&(seg[lag:]==seg[:-lag])[:,None]
        x[lag:,slot:slot+50]=np.where(ok,v[lag:,:50]-v[:-lag,:50],0)
        mask[lag:,slot:slot+50]=ok
    return x,mask

def pool5(v,m,seg):
    n=len(v)//5; k=v.shape[1]
    v=v[:n*5].reshape(n,5,k);m=m[:n*5].reshape(n,5,k).astype(bool)
    same=seg[:n*5].reshape(n,5); okseg=np.all(same==same[:,-1:],1)
    count=m.sum(1); enough=(count>=4)&okseg[:,None]
    last=m[:,-1]&okseg[:,None]
    mean=np.divide(np.where(m,v,0).sum(1,dtype=np.float64),count,out=np.zeros((n,k)),where=count>0)
    maximum=np.where(m,v,-np.inf).max(1)
    out=np.stack((v[:,-1],mean,maximum),-1).reshape(n,3*k).astype(np.float32)
    masks=np.stack((last,enough,enough),-1).reshape(n,3*k)
    out[~masks]=0
    return out,masks


In [ ]:
if HAS_TORCH:
    class NpuFP32GRU(nn.GRU):
        def flatten_parameters(self):
            # No cuDNN/NPU packed-weight storage is needed by the explicit path.
            pass

        def forward(self, input, hx=None):
            if not self.batch_first or self.bidirectional:
                raise ValueError('NpuFP32GRU requires batch_first=True and unidirectional')
            if not isinstance(input, torch.Tensor) or input.ndim != 3:
                raise ValueError('NpuFP32GRU requires a dense [batch,time,features] tensor')
            b, t, width = input.shape
            if not b or not t or width != self.input_size:
                raise ValueError('Invalid NpuFP32GRU input shape')
            if input.dtype != torch.float32 or any(p.dtype != torch.float32 for p in self.parameters()):
                raise ValueError('TDHA GRU must use FP32 inputs and parameters; AMP is not enabled')
            if hx is None:
                hx = input.new_zeros(self.num_layers, b, self.hidden_size)
            if hx.shape != (self.num_layers, b, self.hidden_size) or hx.dtype != input.dtype or hx.device != input.device:
                raise ValueError('Invalid NpuFP32GRU initial hidden state')
            output = input
            final = []
            for layer in range(self.num_layers):
                wi = getattr(self, f'weight_ih_l{layer}')
                wh = getattr(self, f'weight_hh_l{layer}')
                bi = getattr(self, f'bias_ih_l{layer}') if self.bias else None
                bh = getattr(self, f'bias_hh_l{layer}') if self.bias else None
                # Batch the input projection; only the recurrent projection is serial.
                projected = F.linear(output, wi, bi)
                h = hx[layer]
                sequence = []
                for x in projected.unbind(1):
                    ir, iz, inn = x.chunk(3, dim=-1)
                    hr, hz, hn = F.linear(h, wh, bh).chunk(3, dim=-1)
                    reset = torch.sigmoid(ir + hr)
                    update = torch.sigmoid(iz + hz)
                    candidate = torch.tanh(inn + reset * hn)
                    h = candidate + update * (h - candidate)
                    sequence.append(h)
                output = torch.stack(sequence, dim=1)
                final.append(h)
                if layer < self.num_layers - 1 and self.dropout:
                    output = F.dropout(output, p=self.dropout, training=self.training)
            return output, torch.stack(final, dim=0)

    class NpuCausalConv(nn.Conv1d):
        def forward(self,x):return super().forward(F.pad(x,((self.kernel_size[0]-1)*self.dilation[0],0)))

    class NpuWaveEncoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv=nn.Sequential(nn.Conv1d(3,8,15,4,7),nn.ReLU(),nn.Conv1d(8,16,9,4,4),nn.ReLU(),nn.Conv1d(16,16,5,2,2),nn.ReLU())
            self.head=nn.Linear(256,16);self.token=nn.Linear(32,64)
        def cycles(self,x):
            h=self.conv(x).reshape(-1,16,8,4)
            return self.head(torch.cat((h.mean(-1),h.amax(-1)),1).flatten(1))
        def forward(self,x,valid):
            b,t,_,_,_=x.shape
            h=self.cycles(x.reshape(-1,3,1024)).reshape(b,t,5,16)
            h=h*valid.unsqueeze(-1)
            mean=h.sum(2)/valid.sum(2).clamp_min(1).unsqueeze(-1)
            maximum=h.masked_fill(~valid.bool().unsqueeze(-1),-1e20).amax(2)
            ok=valid.sum(2)>=4
            return self.token(torch.cat((mean,maximum),-1))*ok.unsqueeze(-1)

    class NpuDVIEncoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv=nn.Sequential(nn.Conv2d(6,24,3,2,1),nn.ReLU(),nn.Conv2d(24,48,3,2,1),nn.ReLU(),nn.Conv2d(48,64,3,2,1),nn.ReLU())
            self.fc=nn.Linear(128,64)
        def forward(self,x,valid):
            b,t=x.shape[:2];h=self.conv(x.reshape(-1,6,32,32)).flatten(2)
            return self.fc(torch.cat((h.mean(-1),h.amax(-1)),1)).reshape(b,t,64)*valid.unsqueeze(-1)

    class NpuFront(nn.Module):
        def __init__(self,rep):
            super().__init__();self.rep=rep;self.power=nn.Linear(108,64)
            if rep=='F':self.hf=nn.Sequential(nn.Linear(1200,128),nn.ReLU(),nn.Linear(128,64))
            elif rep=='W1':self.hf=NpuWaveEncoder()
            elif rep=='DVI':self.hf=NpuDVIEncoder()
        def forward(self,batch):
            p=self.power(batch['power'])
            if self.rep=='P':h=torch.zeros_like(p)
            elif self.rep=='F':h=self.hf(batch['extra'])*(batch['extra'][...,600:].sum(-1)>0).unsqueeze(-1)
            else:h=self.hf(batch['extra'],batch['valid'])
            return torch.cat((p,h),-1)

    class NpuLargeBlock(nn.Module):
        def __init__(self,d):
            super().__init__();self.large=NpuCausalConv(192,192,31,dilation=d,groups=192);self.small=NpuCausalConv(192,192,3,dilation=d,groups=192)
            self.norm=nn.LayerNorm(192);self.ff=nn.Sequential(nn.Linear(192,768),nn.GELU(),nn.Dropout(.1),nn.Linear(768,192),nn.Dropout(.1))
        def forward(self,x):
            h=(self.large(x.transpose(1,2))+self.small(x.transpose(1,2))).transpose(1,2)
            return x+self.ff(self.norm(h))

    class NpuAttentionBlock(nn.Module):
        """Causal self-attention with residual feed-forward updates."""
        def __init__(self):
            super().__init__();self.ln1=nn.LayerNorm(192);self.qkv=nn.Linear(192,576);self.proj=nn.Linear(192,192)
            self.ln2=nn.LayerNorm(192);self.drop=nn.Dropout(.1)
            self.ff=nn.Sequential(nn.Linear(192,768),nn.GELU(),nn.Dropout(.1),nn.Linear(768,192),nn.Dropout(.1))
        def forward(self,x):
            b,t,_=x.shape;q,k,v=self.qkv(self.ln1(x)).reshape(b,t,3,6,32).permute(2,0,3,1,4).unbind(0)
            a=q@k.transpose(-1,-2)/math.sqrt(32)
            causal=torch.ones((t,t),dtype=torch.bool,device=x.device).triu(1)
            a=self.drop(a.masked_fill(causal,-float('inf')).softmax(-1))
            x=x+self.drop(self.proj((a@v).transpose(1,2).reshape(b,t,192)))
            return x+self.ff(self.ln2(x))

    class NpuBackbone(nn.Module):
        def __init__(self,name):
            super().__init__();self.name=name
            if name=='N0':
                self.conv=NpuCausalConv(128,32,5);self.rnn=NpuFP32GRU(32,48,2,batch_first=True,dropout=.1)
                self.head=nn.Sequential(nn.Linear(48,32),nn.ReLU(),nn.Dropout(.1),nn.Linear(32,11))
            elif name=='N1':
                self.proj=nn.Linear(128,192);self.blocks=nn.Sequential(*[NpuLargeBlock(d) for d in (1,1,2,2,4,4)])
                self.head=nn.Linear(192,11)
            elif name=='N2':
                self.proj=nn.Linear(512,192);self.blocks=nn.Sequential(*[NpuAttentionBlock() for _ in range(4)])
                self.head=nn.Linear(192,11)
                pe=torch.zeros(64,192);pos=torch.arange(64).float()[:,None];freq=torch.exp(torch.arange(0,192,2).float()*(-math.log(10000)/192))
                pe[:,0::2]=torch.sin(pos*freq);pe[:,1::2]=torch.cos(pos*freq);self.register_buffer('pe',pe)
            else:raise ValueError(name)
        def forward(self,x):
            if self.name=='N0':h,_=self.rnn(F.relu(self.conv(x.transpose(1,2))).transpose(1,2))
            elif self.name=='N1':h=self.blocks(self.proj(x))
            else:
                b,t,c=x.shape;assert t%4==0
                h=self.blocks(self.proj(x.reshape(b,t//4,c*4))+self.pe[None,:t//4])
            return self.head(h[:,-1])

    class NpuModel(nn.Module):
        def __init__(self,network,rep,seed):
            super().__init__()
           
            with torch.random.fork_rng(devices=[]):
                torch.manual_seed(seed+10000);self.front=NpuFront(rep)
                torch.manual_seed(seed+20000);self.backbone=NpuBackbone(network)
        def forward(self,batch):return self.backbone(self.front(batch))
else:
    print("PyTorch is required for model construction.")


## 9. Multi-label loss and inference

Class weights are estimated from known training labels. Each active head uses its own known-sample denominator; active heads receive equal weight. Reference inference treats each history window independently, without carrying GRU hidden state between windows.


V3 maintains known and strict masks for each of 11 heads. Invalid states remain unknown until a valid label; transition masks use 150 ms and 500 ms.


In [ ]:
def npu_distance(t, events):
    if not len(events):
        return np.full(len(t), 10 ** 15, np.int64)
    p = np.searchsorted(events, t)
    return np.minimum(np.abs(t - events[np.maximum(p - 1, 0)]), np.abs(t - events[np.minimum(p, len(events) - 1)]))

def npu_project(lab, t):
    ix = np.searchsorted(lab['time'], t, side='right') - 1
    span = (ix >= 0) & (t <= lab['time'][-1])
    y = lab['state'][np.maximum(ix, 0)].copy()
    y[~span] = 0
    dist = np.stack([npu_distance(t, lab['time'][np.r_[False, np.diff(lab['state'][:, c].astype(int)) != 0]]) for c in range(11)], 1)
    if 'state_valid' in lab:
        valid = lab['state_valid']
        state_known = valid[np.maximum(ix, 0)]
        for c in range(11):
            edges = np.r_[not valid[0, c], (np.diff(lab['state'][:, c].astype(int)) != 0) | ~valid[1:, c] | ~valid[:-1, c]]
            dist[:, c] = npu_distance(t, lab['time'][edges])
        return {'y': y, 'known': span[:, None] & state_known & (dist > 150000), 'strict': span[:, None] & state_known & (dist > 500000), 'near': npu_distance(t, lab['any_switch']), 'switch_distance': dist}
    return {'y': y, 'known': span[:, None] & (dist > 150000), 'strict': span[:, None] & (dist > 500000), 'near': npu_distance(t, lab['any_switch']), 'switch_distance': dist}


In [ ]:
def require(condition, message):
    if not condition: raise ValueError(message)

def class_weights(ds):
    pos=np.zeros(11);neg=np.zeros(11)
    for r in np.unique(ds.pairs[:,0]):
        arr,_=ds.arrays(int(r));ix=ds.pairs[ds.pairs[:,0]==r,1];y=arr['y'][ix];m=arr['known'][ix]
        pos+=(y*m).sum(0);neg+=((1-y)*m).sum(0)
    active=(pos>0)&(neg>0);n=pos+neg
    return {'positive':pos,'negative':neg,'active':active,'wp':n/(2*np.maximum(pos,1)),'wn':n/(2*np.maximum(neg,1))}

def balanced_loss(logits,y,mask,weights):
    w=y*weights['wp']+(1-y)*weights['wn'];m=mask*weights['active']
    loss=torch.nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')*w*m
   
    den=m.sum(0);head=den>0
    return (loss.sum(0)/den.clamp_min(1))[head].mean()

def calibration_loss(y,z,mask,active,primary=True):
    losses=np.logaddexp(0,z.astype(np.float64))-y*z
    values=[]
    for c in range(6 if primary else 11):
        on=mask[:,c].astype(bool)&(y[:,c]==1);off=mask[:,c].astype(bool)&(y[:,c]==0)
        if active[c] and on.any() and off.any():values.append((losses[on,c].mean()+losses[off,c].mean())/2)
    require(len(values)==6 if primary else len(values)>0,'Calibration segment lacks computable primary classes')
    return float(np.mean(values))

@manual_step("inference")
def predict_reference(model,ds,device,micro):
    from tdha_v3.training import rng_state, rng_hash, sync, collate
    model.eval();before=rng_state(device);parts=[]
   
    with torch.inference_mode():
        for lo in range(0,len(ds),micro):parts.append(model(collate(ds,range(lo,min(lo+micro,len(ds))),device)).cpu().numpy().astype(np.float32))
    sync(device);require(rng_hash(before)==rng_hash(rng_state(device)),'Evaluation consumed training RNG')
    return np.concatenate(parts) if parts else np.empty((0,11),np.float32)


NPU training uses AdamW (`lr=3e-4`, `weight_decay=1e-4`, `foreach=False`), effective batch size 64, fixed orders and microbatch accumulation. The epoch implementation applies the objective denominator and gradient checks; launch, resume and checkpoint writing are handled by the training module.


In [ ]:
@manual_step("training")
def train_npu_epoch(model, train, trainmeta, orders, epoch, weights, device, micro, optimizer):
    from tdha_v3.memory import BatchReader
    model.train(); losses=[]
    with BatchReader() as reader:
        for lo in range(0,len(train.pairs),64):
            indices=orders[epoch,lo:lo+64]
            optimizer.zero_grad(set_to_none=True)
            ys=trainmeta['mask'][indices]
            den=torch.tensor(ys.sum(0),device=device)*weights['active']; heads=den>0
            groups=(indices[q:q+micro] for q in range(0,len(indices),micro))
            for batch in reader.batches(train,groups,device):
                z=model(batch)
                raw=F.binary_cross_entropy_with_logits(z,batch['y'],reduction='none')
                w=batch['y']*weights['wp']+(1-batch['y'])*weights['wn']
                m=batch['mask']*weights['active']
                loss=((raw*w*m).sum(0)/den.clamp_min(1))[heads].mean()
                require(torch.isfinite(loss).item(),'Non-finite training loss')
                loss.backward(); losses.append(float(loss.detach().cpu()))
            require(all(p.grad is None or torch.isfinite(p.grad).all().item() for p in model.parameters()),'Non-finite gradients')
            optimizer.step()
    return losses


## 10. Checkpoint loading and conversion

CPU loading verifies a trusted SHA-256 before deserialization, uses `weights_only=True`, validates the state container and loads strictly. NPU conversion inspects serialization globals, reconstructs bounded FP32 CPU storage, compares tensor hashes with the conversion record and checks the output by reloading it. The source is never overwritten, and unsafe pickle fallback is unsupported.

[convert_weights.py](../npu/tdha_multilabel_v3/tools/convert_weights.py) contains the complete converter and its Ascend/PyTorch BSD attribution. `RUN_CONVERSION` controls execution.


In [ ]:
@manual_step("weights")
def load_frozen_cpu(checkpoint_path, expected_sha256, model):
    path=Path(checkpoint_path)
    if not path.is_file():
        raise FileNotFoundError("Obtain the selected model attachment or request it by email")
    h=hashlib.sha256()
    with path.open("rb") as f:
        for part in iter(lambda:f.read(8*1024*1024),b""):
            h.update(part)
    if h.hexdigest() != expected_sha256:
        raise ValueError("Checkpoint SHA-256 differs from the trusted manifest")
    checkpoint=torch.load(path,map_location="cpu",weights_only=True)
    state=checkpoint.get("model",checkpoint.get("state_dict",checkpoint)) if isinstance(checkpoint,dict) else checkpoint
    if not isinstance(state,(dict,OrderedDict)) or not state:
        raise ValueError("Invalid state dictionary")
    if not all(isinstance(k,str) and isinstance(v,torch.Tensor) and v.device.type=="cpu" for k,v in state.items()):
        raise ValueError("Only named CPU tensors are accepted")
    model.load_state_dict(state,strict=True)
    model.eval()
    return model

@manual_step("conversion")
def restricted_npu_load(source, expected_sha256):
    # Reconstruct bounded CPU storage without torch_npu or unrestricted pickle.
   
    import importlib.util
    tool_path=ROOT/"npu/tdha_multilabel_v3/tools/convert_weights.py"
    sys.path.insert(0,str(tool_path.parent))
    spec=importlib.util.spec_from_file_location("wattsign_conversion",tool_path)
    conversion=importlib.util.module_from_spec(spec); spec.loader.exec_module(conversion)
    source=Path(source)
    audit=conversion.static_review(source)
    if audit['source_sha256'] != expected_sha256:
        raise ValueError("Source checkpoint does not match the trusted manifest")
    with torch.serialization.safe_globals([
        (conversion.rebuild_cpu_tensor,"torch_npu.utils.storage._rebuild_npu_tensor"),
        (conversion.CPUFormat,"torch_npu.npu._format.Format"),
    ]):
        checkpoint=torch.load(source,map_location="cpu",weights_only=True)
    return checkpoint, audit


## 11. Shared-block model comparison

Paired bootstrap gives records and folds equal weight and shares block weights across models, seeds and heads. AP and Recall exclude absent classes; OFF FPR includes them. This preserves pairing while accounting for temporal dependence. Result summaries use the recorded comparison; execution requires `RUN_BOOTSTRAP`.


In [ ]:
@manual_step("bootstrap")
def block_bootstrap_pair(records,iterations=2000,seed=42):
    """Estimate paired metric deltas using shared temporal block weights."""
    rng=np.random.default_rng(seed);results=[];draws=[];invalid=0
    prepared=[]
    for r in records:
        block=(r['time']-r['time'][0])//60000000;_,inv=np.unique(block,return_inverse=True)
        r['_sorted']={}
        for c in range(6):
            ok=r['mask'][:,c].astype(bool)
            for tag in ('a','b'):
                for si in range(len(r[tag])):
                    z=r[tag][si,ok,c];order=np.argsort(-z,kind='stable');zz=z[order]
                    r['_sorted'][tag,si,c]=(order,np.r_[zz[1:]!=zz[:-1],True] if len(zz) else np.empty(0,bool))
        prepared.append((r,inv,int(inv.max()+1)))
    for attempt in range(iterations*20):
        if len(results)==iterations:break
        per_fold={};draw={};valid_repeat=True
        for ri,(r,inv,k) in enumerate(prepared):
            counts=np.bincount(rng.integers(k,size=k),minlength=k);w=counts[inv];draw[str(ri)]=counts.tolist()
            fold=per_fold.setdefault(r['fold'],{c:[] for c in range(6)})
            for c in range(6):
                if not r['trained'][c]:continue
                ok=r['mask'][:,c].astype(bool);y=r['y'][ok,c];ww=w[ok]
                if not len(y):continue
                if (y==0).any() and (ww*(1-y)).sum()==0:valid_repeat=False
                if (y==1).any() and (ww*y).sum()==0:valid_repeat=False
                values=[]
                for tag,scores,thresholds in (('a',r['a'],r['ta']),('b',r['b'],r['tb'])):
                    seeds=[]
                    for si in range(len(scores)):
                        z=scores[si,ok,c];t=thresholds[si,c];on=ww*y;off=ww*(1-y)
                        order,ends=r['_sorted'][tag,si,c]
                        tp=np.cumsum(ww[order]*y[order])[ends];total=np.cumsum(ww[order])[ends]
                        ap=float(np.sum(np.diff(np.r_[0,tp])*tp/np.maximum(total,1))/tp[-1]) if r['present'][c] and len(tp) and tp[-1]>0 and total[-1]>tp[-1] else np.nan
                        if r['present'][c] and len(np.unique(y))==2 and not np.isfinite(ap):valid_repeat=False
                        if not np.isfinite(t):valid_repeat=False
                        rec=np.sum(on*(z>=t))/on.sum() if on.sum() and r['present'][c] else np.nan
                        fpr=np.sum(off*(z>=t))/off.sum() if off.sum() else np.nan
                        seeds.append([ap,rec,fpr])
                    values.append(np.mean(seeds,axis=0))
                fold[c].append(np.array(values))
        folds=[]
        for f,classes in per_fold.items():
            cs=[]
            for c,vals in classes.items():
                if not vals:cs.append(np.full(3,np.nan));continue
                v=np.stack(vals)
               
                a=np.nanmean(v,axis=0);cs.append(a[1]-a[0])
            folds.append(np.mean(cs,axis=0))
        delta=np.mean(folds,axis=0)
        if valid_repeat and np.isfinite(delta).all():results.append(delta);draws.append(draw)
        else:invalid+=1
    a=np.array(results)
    return {'metrics':['AP','Recall','OFF_FPR'],'valid':len(a),'invalid':invalid,'complete':len(a)==iterations,
            'CI':np.quantile(a,[.025,.975],axis=0).tolist() if len(a) else None,'samples':a,'draws':draws}


## 12. Train from raw data

[tdha_waveform_v2.py](../research/tdha_waveform_v2.py) pairs ADC cycles, projects labels and continuity, fits training-only fold normalization, fixes balanced samples/orders, trains GRU/W1 and calibrates thresholds. It saves weights, time/cross-record/record-1 pressure predictions and per-record metrics. Inputs are local `Aggregate.zip`, `SocketRecord.xlsx` and a new output directory; derived caches are generated within that run.

The default selection includes all six fold/seed runs. Enable `RUN_TRAINING`, `RUN_DATA_READ` and `RUN_INFERENCE` to execute. Run outputs are separate from the recorded experiments; numerical results depend on hardware and dependency versions.

V3 preparation and training are provided by [npu/tdha_multilabel_v3](../npu/tdha_multilabel_v3/README.md). NPU training requires its documented Linux, tmux and NPU environment.


In [ ]:
if RUN_TRAINING:
    if not RUN_DATA_READ or not RUN_INFERENCE:
        raise RuntimeError("Full training requires RUN_DATA_READ and RUN_INFERENCE explicitly")
    if TDHA_ARCHIVE is None or TDHA_MAPPING is None or TRAIN_OUTPUT is None:
        raise FileNotFoundError("Set TDHA_ARCHIVE, TDHA_MAPPING and a fresh TRAIN_OUTPUT")
    research_path = ROOT/"research"
    if str(research_path) not in sys.path:
        sys.path.insert(0,str(research_path))
    from tdha_waveform_v2 import run_selected
    new_runs = run_selected(TDHA_ARCHIVE, TDHA_MAPPING, TRAIN_OUTPUT,
                            device=TRAIN_DEVICE, folds=("A","B"), seeds=SEEDS)
else:
    print("Training: disabled (RUN_TRAINING=False).")


## 13. Results and limits

V2 includes all six fold/seed runs. Its cross-record **AP 0.9843** indicates strong separation of ON and OFF scores, while **Recall 80.23%** means that a substantial share of valid ON endpoints is still missed at the fixed threshold. The result concerns microwave supply state in controlled recordings.

V3 includes all twelve runs. N1-F improves the temporal hold-out results within the recording script used for NPU training, but its AP and Recall fall below N0-W1 on the CUDA cross-script test. It reduces false positives while missing more supplied states, and **does not meet the frozen selection criteria**. The comparison shows why an improvement within one recording procedure needs a separate check under changed appliance combinations and operating sequences.

The hardware labels identify the work actually performed: **NPU** for training and temporal hold-out evaluation, **CUDA** for fixed-weight cross-script and third-record supplemental evaluation, and **CPU format** for portable checkpoint storage and loading. CPU conversion verifies storage compatibility; the available results do not cover a complete NPU evaluation workflow. V2 and V3 also have different targets and histories, so their absolute scores are not directly comparable.

[Results and evaluation](../docs/en/results.md) · [Data](../docs/en/data.md)


## Recorded metrics

**Average precision (AP)** summarizes ON/OFF score ranking. **Recall** measures the share of valid ON endpoints detected at the fixed threshold; **F1** combines precision and recall. OFF false-positive rate (**OFF FPR**) measures erroneous ON predictions among valid OFF endpoints. For multi-label evaluation, the supported labels and averaging rules are defined in the results guide.

Values below are rounded from the case summaries and include all protocol seeds/folds. The results guide defines aggregation, confidence intervals and limitations.

### tdha-waveform

[Source summary](../results/tdha-waveform/summary.json) · [English results](../docs/en/results.md)

| Partition | Model | Backend | AP | Recall | F1 |
|---|---|---|---:|---:|---:|
| Cross-record | GRU/W1/L2560 | — | 0.9843 | 0.8023 | 0.8586 |

### tdha-npu

[Source summary](../results/tdha-npu/summary.json) · [English results](../docs/en/results.md)

| Partition | Model | Backend | AP | Recall | F1 |
|---|---|---|---:|---:|---:|
| time | N0-W1 | NPU | 0.8647 | 0.9087 | 0.6987 |
| cross | N0-W1 | CUDA | 0.7519 | 0.5155 | 0.2948 |
| time | N1-F | NPU | 0.9492 | 0.9314 | 0.7076 |
| cross | N1-F | CUDA | 0.7364 | 0.4063 | 0.2518 |

N1-F did not meet the original frozen decision criteria. NPU temporal hold-out evaluations and CUDA supplemental cross-record tests have different scopes.


In [ ]:
from wattsign.notebook_view import show_experiment
show_experiment("tdha-waveform")
show_experiment("tdha-npu")
